# Fock Attention: Literal §5.1 Feynman Diagram as Exchange Force (H=128)

## Motivation

This notebook implements the literal Section 5.1 Feynman diagram of attention
as a **direct token-to-token non-conservative exchange force** within the
Multi-ξ PARFLM framework.  Unlike Fock v2 (which routes through persistent
registers), Fock Attention uses instantaneous virtual photon exchange:

- Token j emits (key k_j, payload v_j)
- Token i absorbs (query q_i)
- Coupling α_ij = softmax_j(q_i · k_j / √d_k)
- Force F_i = Σ_j α_ij · v_j

This is the λ=0 (instantaneous) limit — no registers, no persistence,
no creation/destruction gates.  The exchange force is injected post-Verlet
using the same pattern as Fock v2's reverse channel.

## Key Questions

| Comparison | What it tests |
|---|---|
| PARFLM vs Fock Attention | Does directed exchange help within the PARF framework? |
| Fock Attention vs Fock v2 | Does temporal persistence (registers, λ>0) add value? |
| Single-head vs Multi-head | Does multi-head exchange improve coupling quality? |

## Baselines (from prior experiments)

| Model | PPL | Steps | Notes |
|-------|-----|-------|-------|
| Attention baseline | 7.81 | 8000 | Target |
| Multi-ξ PARF K=8 (best conservative) | 12.06 | 8000 | Log-spaced α |
| Multi-ξ PARF K=4 | 12.47 | 8000 | Log-spaced α |
| Fock v2 K=4 M=16 LIFO | 14.21 | 8000 | With reverse channel |
| Fock v2 K=4 M=16 LIFO 16k | 12.00 | 16000 | Best Fock result |

## Arms (7-arm sweep)

| # | Arm | K | n_heads | d_k | Steps | Purpose |
|---|-----|---|---------|-----|-------|--------|
| 1 | `direct_K4_h1_8k` | 4 | 1 | 64 | 8000 | Single-head baseline |
| 2 | `direct_K4_h1_16k` | 4 | 1 | 64 | 16000 | Extended convergence |
| 3 | `direct_K4_h4_8k` | 4 | 4 | 32 | 8000 | Multi-head exchange |
| 4 | `direct_K4_h4_16k` | 4 | 4 | 32 | 16000 | Multi-head extended |
| 5 | `direct_K8_h1_8k` | 8 | 1 | 64 | 8000 | More ξ channels |
| 6 | `direct_K8_h4_8k` | 8 | 4 | 32 | 8000 | Best of both |
| 7 | `direct_K4_noforce_8k` | 4 | — | — | 8000 | Control (force=off) |

## Hardware

- **A100 40GB / H100 80GB**: Exchange force adds ~50–130K params overhead.
  No register concatenation, so wall-time is close to plain Multi-ξ PARF.
- No grad-accum needed
- TF32 disabled for autograd.grad stability

## 1. Environment setup

In [ ]:
import os, sys, subprocess, shutil, json, time, math
from pathlib import Path

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

IN_COLAB = 'google.colab' in sys.modules
print('In Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_fock_attention_h128')
    REPO_PARENT = Path('/content')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_fock_attention_h128'
    REPO_PARENT = Path.cwd().parent.parent.parent.parent

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print('Drive root   :', DRIVE_ROOT)
print('Results dir  :', DRIVE_RESULTS)

In [ ]:
REPO_URL = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_DIR = REPO_PARENT / 'semsimula-paper'

if IN_COLAB:
    if REPO_DIR.exists():
        print(f'Repo already cloned at {REPO_DIR}')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       check=False)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL,
                        str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch', 'numpy', 'matplotlib', 'tiktoken', 'datasets'],
                   check=True)

SCRIPTS_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'scaleup'
PARF_DIR    = REPO_DIR / 'notebooks' / 'conservative_arch' / 'parf'
MULTIXI_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'multixi'
assert SCRIPTS_DIR.exists(), f'Missing: {SCRIPTS_DIR}'
print('Scripts dir  :', SCRIPTS_DIR)

## 2. GPU check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    DEVICE = 'cuda'
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for PARF autograd.grad stability')
elif torch.backends.mps.is_available():
    print('GPU: Apple MPS')
    DEVICE = 'mps'
else:
    print('WARNING: No GPU detected')
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 3. Experiment configuration

All arms use the memory-optimized Multi-ξ PARF backbone + direct exchange:
- `--use-layer-checkpoint` (Level-2: O(L) → O(1) layer memory)
- `--use-gathered-v-phi`  (Stage-1.5b: O(T²) → O(T·k) V_φ memory)
- `--v-phi-phi-hidden 128 --v-phi-theta-hidden 128` (full V_φ capacity)
- `--ln-before-distance --per-layer-v-phi-scale` (P8 patches)
- `--xi-alpha-init-mode log_spaced` (auto-tuned α)
- `--gumbel-tau-min 0.3` (raised floor for stability)
- `--exchange-grad-clip 0.5` (separate tighter clip for exchange params)

In [ ]:
MODE            = 'scaleup'
FIXED_GAMMA     = 0.30
SEED            = 0
MAX_TRAIN_TOK   = 5_000_000
V_PHI_H         = 128
GUMBEL_TAU_MIN  = 0.3
EX_GRAD_CLIP    = 0.5

ARM_DEFS = {
    # ---- Single-head arms ----
    'direct_K4_h1_8k': {
        'exchange_n_heads': 1,
        'exchange_d_k': 64,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'max_steps': 8000,
        'desc': 'K=4, 1-head dk=64, 8k steps — single-head baseline',
    },
    'direct_K4_h1_16k': {
        'exchange_n_heads': 1,
        'exchange_d_k': 64,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'max_steps': 16000,
        'desc': 'K=4, 1-head dk=64, 16k steps — extended convergence',
    },
    # ---- Multi-head arms ----
    'direct_K4_h4_8k': {
        'exchange_n_heads': 4,
        'exchange_d_k': 32,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'max_steps': 8000,
        'desc': 'K=4, 4-head dk=32, 8k steps — multi-head exchange',
    },
    'direct_K4_h4_16k': {
        'exchange_n_heads': 4,
        'exchange_d_k': 32,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'max_steps': 16000,
        'desc': 'K=4, 4-head dk=32, 16k steps — multi-head extended',
    },
    # ---- K=8 arms ----
    'direct_K8_h1_8k': {
        'exchange_n_heads': 1,
        'exchange_d_k': 64,
        'xi_channels': 8,
        'xi_alpha_init_mode': 'log_spaced',
        'max_steps': 8000,
        'desc': 'K=8, 1-head dk=64, 8k steps — more ξ channels',
    },
    'direct_K8_h4_8k': {
        'exchange_n_heads': 4,
        'exchange_d_k': 32,
        'xi_channels': 8,
        'xi_alpha_init_mode': 'log_spaced',
        'max_steps': 8000,
        'desc': 'K=8, 4-head dk=32, 8k steps — best of both',
    },
    # ---- Control arm (no exchange force) ----
    'direct_K4_noforce_8k': {
        'exchange_n_heads': 1,
        'exchange_d_k': 64,
        'xi_channels': 4,
        'xi_alpha_init_mode': 'log_spaced',
        'max_steps': 8000,
        'disable_exchange': True,
        'desc': 'K=4, exchange=OFF, 8k steps — control (pure MultiXi K4)',
    },
}

ALL_ARMS = list(ARM_DEFS.keys())

# To run only specific arms, edit this list:
ARMS_TO_RUN = ALL_ARMS
# ARMS_TO_RUN = ['direct_K4_h1_8k', 'direct_K4_h4_8k']

print(f'Arms to run: {ARMS_TO_RUN}')
for name in ARMS_TO_RUN:
    print(f'  {name}: {ARM_DEFS[name]["desc"]}')

## 4. Training loop with streaming output

In [ ]:
import re
from IPython import display

_TRAIN_RE = re.compile(
    r'\[fock-attention\]\s+step\s+(\d+)/(\d+)\s+train\s+([\d.]+).*?elapsed\s+([\d.]+)s'
)
_EVAL_RE = re.compile(
    r'\[fock-attention\]\s+>>>\s+eval\s+@\s+(\d+):\s+val\s+([\d.]+)\s+ppl\s+([\d.]+)'
)


def _build_arm_cmd(arm_name: str, arm_cfg: dict, arm_results_dir: Path) -> list[str]:
    """Build the CLI command for train_fock_attention_scaleup.py."""
    cmd = [
        sys.executable, str(SCRIPTS_DIR / 'train_fock_attention_scaleup.py'),
        '--mode', MODE,
        '--fixed-gamma', str(FIXED_GAMMA),
        '--seed', str(SEED),
        '--max-train-tokens', str(MAX_TRAIN_TOK),
        '--device', DEVICE,
        '--use-layer-checkpoint',
        '--use-gathered-v-phi',
        '--v-phi-phi-hidden', str(V_PHI_H),
        '--v-phi-theta-hidden', str(V_PHI_H),
        '--ln-before-distance',
        '--per-layer-v-phi-scale',
        '--gumbel-tau-min', str(GUMBEL_TAU_MIN),
        '--results-dir', str(arm_results_dir),
        '--tag-suffix', arm_name,
        '--top-k', '8',
        '--xi-channels', str(arm_cfg['xi_channels']),
        '--xi-alpha-init-mode', arm_cfg['xi_alpha_init_mode'],
        '--exchange-n-heads', str(arm_cfg['exchange_n_heads']),
        '--exchange-d-k', str(arm_cfg['exchange_d_k']),
    ]
    if 'xi_alpha_inits' in arm_cfg:
        cmd += ['--xi-alpha-inits', arm_cfg['xi_alpha_inits']]
    if arm_cfg.get('max_steps'):
        cmd += ['--max-steps', str(arm_cfg['max_steps'])]
    if EX_GRAD_CLIP is not None:
        cmd += ['--exchange-grad-clip', str(EX_GRAD_CLIP)]
    if arm_cfg.get('disable_exchange'):
        cmd += ['--disable-exchange']
    return cmd


def _run_arm_streaming(arm_name: str, arm_cfg: dict):
    """Run one arm with live streaming progress."""
    arm_results = DRIVE_RESULTS / arm_name
    arm_results.mkdir(parents=True, exist_ok=True)

    # Skip if already complete
    existing_summaries = list(arm_results.glob('*_summary.md'))
    if existing_summaries:
        print(f'\n=== {arm_name}: already complete, skipping ===')
        print(f'    Summary: {existing_summaries[0].name}')
        return

    cmd = _build_arm_cmd(arm_name, arm_cfg, arm_results)
    print(f'\n{"="*78}')
    print(f'  ARM: {arm_name}')
    print(f'  {arm_cfg["desc"]}')
    print(f'{"="*78}')

    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'

    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env,
    )

    total_steps = arm_cfg.get('max_steps', 8000)
    last_step = 0
    last_train = 0.0
    last_ppl = None
    t_start = time.time()
    all_ppls = []

    for line in proc.stdout:
        line = line.rstrip()

        m_train = _TRAIN_RE.search(line)
        m_eval = _EVAL_RE.search(line)

        if m_train:
            last_step = int(m_train.group(1))
            last_train = float(m_train.group(3))
        elif m_eval:
            eval_step = int(m_eval.group(1))
            eval_ppl = float(m_eval.group(3))
            last_ppl = eval_ppl
            last_step = eval_step
            all_ppls.append((eval_step, eval_ppl))

        if m_train or m_eval:
            elapsed = time.time() - t_start
            frac = last_step / total_steps if total_steps else 0
            eta = (elapsed / frac - elapsed) if frac > 0.01 else 0
            bar_len = 40
            filled = int(bar_len * frac)
            bar = '\u2588' * filled + '\u2591' * (bar_len - filled)
            ppl_str = f'  val PPL: {last_ppl:.2f}' if last_ppl else ''
            display.clear_output(wait=True)
            print(f'ARM: {arm_name} — {arm_cfg["desc"]}')
            print(f'[{bar}] {last_step}/{total_steps}  '
                  f'({100*frac:.1f}%)  '
                  f'train_loss: {last_train:.4f}{ppl_str}  '
                  f'elapsed: {elapsed:.0f}s  ETA: {eta:.0f}s')
            if all_ppls:
                print(f'PPL history: {" \u2192 ".join(f"{p:.2f}" for _, p in all_ppls[-8:])}')
        else:
            # Print non-progress lines (param counts, probes, etc.)
            if line.strip():
                print(line)

    proc.wait()
    if proc.returncode != 0:
        print(f'\nERROR: {arm_name} exited with code {proc.returncode}')
    else:
        print(f'\n{arm_name}: DONE')
        if all_ppls:
            best_step, best_ppl = min(all_ppls, key=lambda x: x[1])
            print(f'  Best val PPL: {best_ppl:.2f} @ step {best_step}')


# Run all arms
for arm_name in ARMS_TO_RUN:
    _run_arm_streaming(arm_name, ARM_DEFS[arm_name])

## 5. Results aggregation

In [ ]:
import matplotlib.pyplot as plt

report = {'experiment': 'fock_attention_h128', 'arms': {}}

for arm_name in ARM_DEFS:
    arm_dir = DRIVE_RESULTS / arm_name
    summaries = list(arm_dir.glob('*_summary.md'))
    logs = list(arm_dir.glob('*_training_log.jsonl'))
    if not summaries:
        report['arms'][arm_name] = {'status': 'not_run'}
        continue

    # Parse log for val PPL history
    ppls = []
    if logs:
        with logs[0].open() as f:
            for line in f:
                entry = json.loads(line)
                if 'val_ppl' in entry:
                    ppls.append((entry['step'], entry['val_ppl']))

    best_ppl = min((p for _, p in ppls), default=None)
    best_step = next((s for s, p in ppls if p == best_ppl), None) if best_ppl else None
    final_ppl = ppls[-1][1] if ppls else None

    report['arms'][arm_name] = {
        'status': 'complete',
        'best_ppl': best_ppl,
        'best_step': best_step,
        'final_ppl': final_ppl,
        'ppl_history': ppls,
        'desc': ARM_DEFS[arm_name]['desc'],
    }

# Save report
report_path = DRIVE_RESULTS / 'fock_attention_h128_report.json'
with report_path.open('w') as f:
    json.dump(report, f, indent=2)
print(f'Report saved: {report_path}')

# Print summary table
print(f'\n{"="*78}')
print(f'{"Arm":30s}  {"Best PPL":>10s}  {"@ Step":>8s}  {"Final PPL":>10s}')
print(f'{"-"*78}')
for arm_name, info in report['arms'].items():
    if info['status'] != 'complete':
        print(f'{arm_name:30s}  {"(not run)":>10s}')
        continue
    print(f'{arm_name:30s}  {info["best_ppl"]:10.2f}  {info["best_step"]:8d}  {info["final_ppl"]:10.2f}')
print(f'{"-"*78}')
print(f'  Prior baselines: MultiXi K8={12.06:.2f}  MultiXi K4={12.47:.2f}  '
      f'Fock v2 16k={12.00:.2f}  Attention={7.81:.2f}')

## 6. Comparison plots

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

colors = {
    'direct_K4_h1_8k': '#1f77b4',
    'direct_K4_h1_16k': '#ff7f0e',
    'direct_K4_h4_8k': '#2ca02c',
    'direct_K4_h4_16k': '#d62728',
    'direct_K8_h1_8k': '#9467bd',
    'direct_K8_h4_8k': '#8c564b',
    'direct_K4_noforce_8k': '#7f7f7f',
}

for arm_name, info in report['arms'].items():
    if info['status'] != 'complete' or not info.get('ppl_history'):
        continue
    steps = [s for s, _ in info['ppl_history']]
    ppls = [p for _, p in info['ppl_history']]
    color = colors.get(arm_name, 'black')
    ax.plot(steps, ppls, marker='o', markersize=3, label=arm_name, color=color)

# Reference lines
ax.axhline(y=12.06, color='green', linestyle='--', alpha=0.5, label='MultiXi K8 (12.06)')
ax.axhline(y=12.47, color='olive', linestyle=':', alpha=0.5, label='MultiXi K4 (12.47)')
ax.axhline(y=12.00, color='red', linestyle='--', alpha=0.5, label='Fock v2 16k (12.00)')
ax.axhline(y=7.81, color='blue', linestyle='--', alpha=0.5, label='Attention (7.81)')

ax.set_xlabel('Step')
ax.set_ylabel('Val PPL')
ax.set_title('Fock Attention (§5.1 Direct Exchange) vs Prior Results')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=6)
fig.tight_layout()

png_path = DRIVE_RESULTS / 'fock_attention_h128_comparison.png'
fig.savefig(png_path, dpi=150)
plt.show()
print(f'Comparison plot saved: {png_path}')

## 7. Convergence analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Exchange scale evolution
ax = axes[0]
for arm_name, info in report['arms'].items():
    if info['status'] != 'complete':
        continue
    arm_dir = DRIVE_RESULTS / arm_name
    logs = list(arm_dir.glob('*_training_log.jsonl'))
    if not logs:
        continue
    steps_ex, scales = [], []
    with logs[0].open() as f:
        for line in f:
            entry = json.loads(line)
            if 'exchange_scale' in entry:
                steps_ex.append(entry['step'])
                scales.append(entry['exchange_scale'])
    if steps_ex:
        color = colors.get(arm_name, 'black')
        ax.plot(steps_ex, scales, label=arm_name, color=color, alpha=0.7)
ax.set_xlabel('Step')
ax.set_ylabel('tanh(exchange_scale)')
ax.set_title('Exchange Force Gate Evolution')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Right: 8k vs 16k PPL comparison
ax = axes[1]
pairs = [
    ('direct_K4_h1_8k', 'direct_K4_h1_16k'),
    ('direct_K4_h4_8k', 'direct_K4_h4_16k'),
]
bar_data = []
labels = []
for short, long in pairs:
    for arm in (short, long):
        info = report['arms'].get(arm, {})
        if info.get('best_ppl'):
            bar_data.append(info['best_ppl'])
            labels.append(arm.replace('direct_', '').replace('_', ' '))

if bar_data:
    x = range(len(bar_data))
    ax.bar(x, bar_data, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(bar_data)])
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Best Val PPL')
    ax.set_title('8k vs 16k Convergence')
    for i, v in enumerate(bar_data):
        ax.text(i, v + 0.1, f'{v:.2f}', ha='center', fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

fig.tight_layout()

conv_path = DRIVE_RESULTS / 'fock_attention_h128_convergence.png'
fig.savefig(conv_path, dpi=150)
plt.show()
print(f'Convergence plot saved: {conv_path}')